In [ ]:
# Diagnostic: confirm notebook kernel and PyTorch
import sys
import torch 
print('Notebook diagnostic')
print('sys.executable:', sys.executable)
print('sys.prefix:', sys.prefix)
print('torch version:', getattr(torch, '__version__', 'not installed'))


ModuleNotFoundError: No module named 'torch'

In [ ]:
# ============================================================
# IoT Attack Classifier (MLP, PyTorch)
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

# ============================================================
# Config
# ============================================================

# ----- paths / columns -----
DATA_PATH = "data/selected_features.csv"   # your feature-selected CSV (49 features)
LABEL_COL = "Label"                        # adjust if your label column has a different name
EXCLUDE_COLS = []                          # any extra non-feature columns to drop (IP, timestamp, device id, ...)

# ----- split -----
TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_STATE = 42

# ----- model / training -----
HIDDEN_DIMS = [256, 128, 64]
DROPOUT = 0.3
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 50
PATIENCE = 7                               # early-stopping patience, in epochs

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

# ============================================================
# Load data
# ============================================================

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df[LABEL_COL].value_counts())
df.head()

# ============================================================
# Preprocess
# ============================================================

feature_cols = [c for c in df.columns if c != LABEL_COL and c not in EXCLUDE_COLS]

non_numeric = df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f"dropping non-numeric feature columns: {non_numeric}")
    feature_cols = [c for c in feature_cols if c not in non_numeric]

X = df[feature_cols].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)  # flow-feature extraction can leave inf/nan

le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL].values)
class_names = le.classes_
n_classes = len(class_names)
print(f"{n_classes} classes: {list(class_names)}")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(TEST_SIZE + VAL_SIZE), stratify=y, random_state=RANDOM_STATE
)
val_ratio = VAL_SIZE / (TEST_SIZE + VAL_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=(1 - val_ratio), stratify=y_temp, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

# ============================================================
# Dataset / DataLoader
# ============================================================

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TabularDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TabularDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# Model
# ============================================================

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_classes, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLPClassifier(X_train.shape[1], HIDDEN_DIMS, n_classes, DROPOUT).to(DEVICE)
print(model)

# ============================================================
# Loss, optimizer
# ============================================================

class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(n_classes), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

# ============================================================
# Train
# ============================================================

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
    return total_loss / n, correct / n

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
epochs_no_improve = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"epoch {epoch:3d} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

# ============================================================
# Evaluate on test set
# ============================================================

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        preds = model(xb).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)

acc = accuracy_score(all_true, all_preds)
macro_f1 = f1_score(all_true, all_preds, average="macro")
weighted_f1 = f1_score(all_true, all_preds, average="weighted")
print(f"test accuracy: {acc:.4f}")
print(f"macro F1:      {macro_f1:.4f}")
print(f"weighted F1:   {weighted_f1:.4f}\n")

print(classification_report(all_true, all_preds, target_names=[str(c) for c in class_names]))

cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (row-normalized)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# ============================================================
# Save model
# ============================================================

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,
    "label_classes": class_names,
    "hidden_dims": HIDDEN_DIMS,
    "input_dim": X_train.shape[1],
}, "attack_classifier_mlp.pt")
print("saved to attack_classifier_mlp.pt")

ModuleNotFoundError: No module named 'torch'